In [ ]:
import torch
from torch import nn as nn
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.ticker import LinearLocator
from mpl_toolkits.mplot3d import axes3d
import scipy.io
from torch.utils.data import DataLoader, random_split, TensorDataset
import pandas as pd
import time

from utils import GiveMeDataMuscleAbove, GiveMeDataMuscleBelow, CreateTensorLoader, plotplot, train_loop, test_loop, plotLossEpoch, plotSemilogyLossEpoch, train_loop_FA, test_loop_FA
%matplotlib inline
%matplotlib widget

In [ ]:
torch.manual_seed(66)
torch.cuda.manual_seed(66)
torch.backends.cudnn.deterministic = True
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.net=nn.Sequential(
            nn.Linear(4,128,dtype=torch.float32),
            nn.Softplus(),
            nn.Linear(128,128,dtype=torch.float32),
            nn.Softplus(),
            nn.Linear(128,1,dtype=torch.float32),
            )
    def forward(self, x):
        return self.net(x)

In [ ]:
file_path = r"C:\\Users\\gwang\\OneDrive - INSA Lyon\\code Python\\FastAdaptation\\code\\data_bm\\sinus_bigmuscle_0g.mat"
# Load the .mat file
mat = scipy.io.loadmat(file_path)
data_train = mat['data']

train_ratio = .8
batch_size = 1024
seed = 66
gpu = 1
e_h, deh_dt, P_h, dPdt_h, q_h = GiveMeDataMuscleAbove(data_train)
train_loader, test_loader = CreateTensorLoader(e_h, deh_dt, P_h, dPdt_h, q_h, 10, train_ratio, batch_size, seed, gpu)
# plotplot(data_train)

In [ ]:
net_bm = Net()
net_bm.to('cuda') if gpu else net_bm.to('cpu')

In [ ]:
loss_function = nn.MSELoss()
optim_bm = torch.optim.Adam(Net.parameters(net_bm), lr=1e-4, weight_decay=1e-5)

In [ ]:
Loss_train_bm = []
Loss_test_bm =[]

In [13]:
print("Model and data is currently on:", next(net_bm.parameters()).device)
print(f"Optimizer = {type(optim_bm).__name__}\nLoss Function = {type(loss_function).__name__}")

target_loss=.001
start_time = time.time()
for t in range(10000):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loss = train_loop(train_loader, net_bm, loss_function, optim_bm, Loss_train_bm, batch_size)
    test_loss = test_loop(test_loader, net_bm, loss_function)
    Loss_test_bm.append(test_loss)
    if test_loss <= target_loss and train_loss <= target_loss:
        print(test_loss)
        break

end_time = time.time()
execution_time = end_time - start_time
execution_time

Model and data is currently on: cuda:0
Optimizer = Adam
Loss Function = MSELoss
Epoch 1
-------------------------------
loss: 235.116653  [ 1024/31464]
loss: 252.714386  [11264/31464]
loss: 217.556335  [21504/31464]
loss: 334.650452  [31464/31464]
Avg training loss: 268.332684
Avg Testing loss: 241.524687 

Epoch 2
-------------------------------
loss: 229.417664  [ 1024/31464]
loss: 303.575226  [11264/31464]
loss: 293.805054  [21504/31464]
loss: 202.768509  [31464/31464]
Avg training loss: 268.332062
Avg Testing loss: 241.524084 

Epoch 3
-------------------------------
loss: 291.653015  [ 1024/31464]
loss: 299.212524  [11264/31464]
loss: 245.995026  [21504/31464]
loss: 260.153320  [31464/31464]
Avg training loss: 268.331280
Avg Testing loss: 241.523300 

Epoch 4
-------------------------------
loss: 261.213440  [ 1024/31464]
loss: 357.323059  [11264/31464]
loss: 219.157623  [21504/31464]
loss: 305.234772  [31464/31464]
Avg training loss: 268.330243
Avg Testing loss: 241.522251 

Epoc

In [ ]:
plotLossEpoch(Loss_train_bm, Loss_test_bm, 0, 'loss_bm.svg')
plotSemilogyLossEpoch(Loss_train_bm, Loss_test_bm, 0, 'semilogy_loss_bm.svg')

In [ ]:
torch.save(net_bm.state_dict(), f"net_bm.pth")
np.save(f"loss_bm_train.npy", Loss_train_bm)
np.save(f"loss_bm_test.npy", Loss_test_bm)